# Chapter 2 — Newtonian Gravitation and Potential Theory

**Galactic Dynamics · Sejong University · 2026B**

This notebook accompanies Chapter 2 of the lecture notes.  You will:

- visualise the Kepler, Plummer, logarithmic, and harmonic potentials and their rotation curves;
- verify Poisson's equation numerically for the Plummer sphere;
- build a composite two-component potential by superposition.

All calculations use **code units** with $G = 1$.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from galdyn.potentials import (
    kepler_potential, kepler_dphi_dr,
    plummer_potential, plummer_density, plummer_enclosed_mass, plummer_dphi_dr,
    logarithmic_potential, logarithmic_dphi_dr,
    harmonic_potential, harmonic_dphi_dr,
)

plt.rcParams.update({
    "figure.dpi": 120,
    "axes.spines.top": False,
    "axes.spines.right": False,
})


---

## 1  The point-mass (Kepler) potential

For a point mass $M$ at the origin,

$$
\Phi(r) = -\frac{GM}{r}, \qquad
g(r) \equiv \frac{d\Phi}{dr} = \frac{GM}{r^2}, \qquad
v_c(r) = \sqrt{r\,g(r)} = \sqrt{\frac{GM}{r}}.
$$

All three are power laws in $r$.


In [ ]:
G, M = 1.0, 1.0
r = np.linspace(0.1, 5.0, 500)

phi  = kepler_potential(r, G=G, M=M)
dphi = kepler_dphi_dr(r, G=G, M=M)
vc   = np.sqrt(r * dphi)

fig, axes = plt.subplots(1, 3, figsize=(12, 3.5))

axes[0].plot(r, phi,  color="steelblue")
axes[0].set_xlabel(r"$r$  [code units]")
axes[0].set_ylabel(r"$\Phi(r)$")
axes[0].set_title("Gravitational potential")

axes[1].plot(r, dphi, color="tomato")
axes[1].set_xlabel(r"$r$")
axes[1].set_ylabel(r"$d\Phi/dr$")
axes[1].set_title("Radial force per unit mass")

axes[2].plot(r, vc,   color="seagreen")
axes[2].set_xlabel(r"$r$")
axes[2].set_ylabel(r"$v_c$")
axes[2].set_title("Circular speed")

fig.suptitle("Kepler potential  ($G = M = 1$)", y=1.02)
fig.tight_layout()
plt.show()


**Exercise 1.1** — Verify that $v_c \propto r^{-1/2}$ by replotting $v_c$ vs $r$ on log–log axes
and overlaying a reference line $\propto r^{-1/2}$.

**Exercise 1.2** — Estimate the orbital period of the Sun around the Galactic Centre.
Assume $v_c = 220\ \text{km s}^{-1}$ at $R_0 = 8\ \text{kpc}$.
Express the answer in Myr.


---

## 2  The Plummer sphere: a potential–density pair

The Plummer model replaces the singular point mass with a softened core of scale $b$:

$$
\Phi(r) = -\frac{GM}{\sqrt{r^2 + b^2}}, \qquad
\rho(r) = \frac{3M}{4\pi b^3}\left(1 + \frac{r^2}{b^2}\right)^{-5/2}.
$$

The enclosed mass is $M(<r) = M\,r^3/(r^2+b^2)^{3/2}$.

### 2.1  Profiles


In [ ]:
G, M, b = 1.0, 1.0, 0.3

r_log = np.logspace(-2, 1, 600)

rho   = plummer_density(r_log, M=M, b=b)
M_enc = plummer_enclosed_mass(r_log, M=M, b=b)

r_lin = np.linspace(0.02, 5.0, 800)
phi_pl  = plummer_potential(r_lin, G=G, M=M, b=b)
phi_kep = kepler_potential(r_lin, G=G, M=M)
dphi_pl  = plummer_dphi_dr(r_lin, G=G, M=M, b=b)
dphi_kep = kepler_dphi_dr(r_lin, G=G, M=M)

# Half-mass radius
r_h = r_log[np.argmin(np.abs(M_enc - 0.5 * M))]

fig, axes = plt.subplots(2, 2, figsize=(10, 8))

# Density (log–log)
ax = axes[0, 0]
ax.loglog(r_log, rho, color="steelblue", label="Plummer")
ax.loglog(r_log, rho[0] * (r_log / r_log[0])**(-5), "k--", lw=1,
          label=r"$\propto r^{-5}$")
ax.axvline(b, color="gray", ls=":", lw=1.2, label=f"$b = {b}$")
ax.set_xlabel(r"$r$")
ax.set_ylabel(r"$\rho(r)$")
ax.set_title("Density")
ax.legend(fontsize=8)

# Enclosed mass
ax = axes[0, 1]
ax.semilogx(r_log, M_enc / M, color="steelblue")
ax.axhline(0.5, color="gray", ls="--", lw=1)
ax.axvline(r_h, color="gray", ls=":",  lw=1.2,
           label=rf"$r_h \approx {r_h:.3f}$  (theory: {1.305*b:.3f})")
ax.set_xlabel(r"$r$")
ax.set_ylabel(r"$M(<r)\ /\ M$")
ax.set_title("Enclosed mass fraction")
ax.legend(fontsize=8)

# Potential: Plummer vs Kepler
ax = axes[1, 0]
ax.plot(r_lin, phi_pl,  color="steelblue", label=f"Plummer ($b={b}$)")
ax.plot(r_lin, phi_kep, "k--", lw=1,       label="Kepler (point mass)")
ax.axvline(b, color="gray", ls=":", lw=1.2)
ax.set_xlabel(r"$r$")
ax.set_ylabel(r"$\Phi(r)$")
ax.set_title("Potential")
ax.legend(fontsize=8)

# Radial force
ax = axes[1, 1]
ax.plot(r_lin, dphi_pl,  color="steelblue", label=f"Plummer ($b={b}$)")
ax.plot(r_lin, dphi_kep, "k--", lw=1,       label="Kepler")
ax.axvline(b, color="gray", ls=":", lw=1.2)
ax.set_xlabel(r"$r$")
ax.set_ylabel(r"$d\Phi/dr$")
ax.set_title("Radial force per unit mass")
ax.legend(fontsize=8)

fig.suptitle(f"Plummer sphere  ($G = M = 1$, $b = {b}$)", y=1.01)
fig.tight_layout()
plt.show()

print(f"Half-mass radius: r_h ≈ {r_h:.4f}    theoretical 1.305 b = {1.305*b:.4f}")


### 2.2  Verifying Poisson's equation numerically

In spherical symmetry Poisson's equation reduces to

$$
\nabla^2\Phi = \frac{1}{r^2}\frac{d}{dr}\!\left(r^2\frac{d\Phi}{dr}\right) = 4\pi G\rho.
$$

We evaluate the left-hand side numerically using `numpy.gradient`.


In [ ]:
# Fine linear grid — np.gradient is most accurate on uniform spacing
r_p = np.linspace(0.02, 8.0, 4000)

f         = r_p**2 * plummer_dphi_dr(r_p, G=G, M=M, b=b)  # r² dΦ/dr
df_dr     = np.gradient(f, r_p)                             # d/dr(r² dΦ/dr)
lap_num   = df_dr / r_p**2                                  # ∇²Φ  (numerical)

rho_p         = plummer_density(r_p, M=M, b=b)
four_pi_G_rho = 4.0 * np.pi * G * rho_p                    # 4πGρ  (analytical)

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

ax = axes[0]
ax.plot(r_p, four_pi_G_rho, color="steelblue", lw=2,
        label=r"$4\pi G\rho$  (analytical)")
ax.plot(r_p, lap_num, "r--", lw=1.5,
        label=r"$\nabla^2\Phi$  (numerical)")
ax.set_xlim(0, 5)
ax.set_xlabel(r"$r$")
ax.set_ylabel(r"$\nabla^2\Phi$")
ax.set_title("Poisson's equation check")
ax.legend()

ax = axes[1]
mask    = (r_p > 0.1) & (r_p < 7.0)
rel_err = np.abs(lap_num[mask] - four_pi_G_rho[mask]) / (four_pi_G_rho[mask] + 1e-14)
ax.semilogy(r_p[mask], rel_err, color="tomato")
ax.set_xlabel(r"$r$")
ax.set_ylabel("relative error")
ax.set_title(r"$\left|\nabla^2\Phi - 4\pi G\rho\right| / 4\pi G\rho$")

fig.tight_layout()
plt.show()


**Exercise 2.1** — Apply the same Poisson check to the Kepler potential for $r > 0.5$.
What do you find?  What is the physical source density for a point-mass potential?

**Exercise 2.2** — The Plummer density falls as $\rho \propto r^{-5}$ for $r \gg b$.
Use this to show that the total mass $M = \int_0^\infty 4\pi r^2\rho\,dr$ converges.
Would a $\rho \propto r^{-3}$ profile be normalisable?

**Exercise 2.3** — Find the radius $r_\star$ at which $d\Phi/dr$ peaks.
*(Hint: set $d^2\Phi/dr^2 = 0$ and solve for $r$.)*
Verify your answer against the plot above.


---

## 3  Comparing potential models

| Model | $\Phi(r)$ | $v_c(r)$ |
|---|---|---|
| Kepler | $-GM/r$ | $\sqrt{GM/r}$ — Keplerian decline |
| Plummer | $-GM/\sqrt{r^2+b^2}$ | rises inside $b$, Keplerian outside |
| Logarithmic | $\tfrac{1}{2}v_0^2\ln(r_c^2+r^2)$ | approaches $v_0$ (flat) |
| Harmonic | $\tfrac{1}{2}\omega_0^2 r^2$ | $\omega_0 r$ — solid-body |

Potentials are defined only up to an additive constant, so we shift each
so that $\Phi(r_\text{ref}) = 0$ at $r_\text{ref} = 2$ for visual comparison.


In [ ]:
r = np.linspace(0.02, 4.5, 800)

models = {
    "Kepler  ($GM=1$)":
        {"phi":  kepler_potential(r, G=1.0, M=1.0),
         "dphi": kepler_dphi_dr(r, G=1.0, M=1.0),
         "color": "steelblue"},
    "Plummer  ($b=0.3$)":
        {"phi":  plummer_potential(r, G=1.0, M=1.0, b=0.3),
         "dphi": plummer_dphi_dr(r, G=1.0, M=1.0, b=0.3),
         "color": "seagreen"},
    "Logarithmic  ($v_0=1,\, r_c=0.3$)":
        {"phi":  logarithmic_potential(r, v0=1.0, rc=0.3),
         "dphi": logarithmic_dphi_dr(r, v0=1.0, rc=0.3),
         "color": "tomato"},
    "Harmonic  ($\\omega_0=0.9$)":
        {"phi":  harmonic_potential(r, omega0=0.9),
         "dphi": harmonic_dphi_dr(r, omega0=0.9),
         "color": "darkorchid"},
}

# Shift Phi so Phi(r=2) = 0
i_ref = np.argmin(np.abs(r - 2.0))
for m in models.values():
    m["phi"] = m["phi"] - m["phi"][i_ref]

fig, axes = plt.subplots(1, 3, figsize=(14, 4.5))

for label, m in models.items():
    vc = np.sqrt(np.maximum(r * m["dphi"], 0.0))
    axes[0].plot(r, m["phi"],  color=m["color"], label=label)
    axes[1].plot(r, m["dphi"], color=m["color"])
    axes[2].plot(r, vc,        color=m["color"])

axes[0].set_ylabel(r"$\Phi(r)$ [shifted: $\Phi(2)=0$]")
axes[0].set_title("Potential")
axes[0].legend(fontsize=7, loc="lower right")

axes[1].set_ylabel(r"$d\Phi/dr$")
axes[1].set_title("Radial force per unit mass")
axes[1].set_ylim(0, 3.5)

axes[2].set_ylabel(r"$v_c = \sqrt{r\,d\Phi/dr}$")
axes[2].set_title("Circular speed")

for ax in axes:
    ax.set_xlabel(r"$r$  [code units]")
    ax.set_xlim(0, 4.5)

fig.tight_layout()
plt.show()


**Exercise 3.1** — The logarithmic potential with $r_c \to 0$ gives a perfectly flat rotation
curve.  Set `rc=1e-3` and confirm.  What density profile does this imply?
*(Use the Poisson technique from Section 2.)*

**Exercise 3.2** — The harmonic potential corresponds to a uniform-density sphere.
Show analytically that $\nabla^2\Phi = 4\pi G\rho_0$ gives $\rho_0 = \omega_0^2/(4\pi G)$.
Verify numerically.

**Exercise 3.3** — Which of the four circular-speed curves could plausibly describe
the outer regions of a real spiral galaxy?  Justify your answer.


---

## 4  Superposition: a bulge–halo model

Because Poisson's equation is linear, potentials add:

$$
\Phi_\text{total} = \Phi_\text{bulge} + \Phi_\text{halo}, \qquad
\rho_\text{total} = \rho_\text{bulge} + \rho_\text{halo}.
$$

We model a galaxy as two Plummer spheres: a compact bulge and an extended halo.


In [ ]:
G = 1.0

M_bulge, b_bulge = 0.2, 0.15   # compact bulge
M_halo,  b_halo  = 0.8, 1.5    # extended halo

r = np.linspace(0.01, 6.0, 800)

phi_bulge  = plummer_potential(r, G=G, M=M_bulge, b=b_bulge)
phi_halo   = plummer_potential(r, G=G, M=M_halo,  b=b_halo)
phi_total  = phi_bulge + phi_halo

dphi_bulge = plummer_dphi_dr(r, G=G, M=M_bulge, b=b_bulge)
dphi_halo  = plummer_dphi_dr(r, G=G, M=M_halo,  b=b_halo)
dphi_total = dphi_bulge + dphi_halo

vc_bulge  = np.sqrt(np.maximum(r * dphi_bulge, 0.0))
vc_halo   = np.sqrt(np.maximum(r * dphi_halo,  0.0))
vc_total  = np.sqrt(np.maximum(r * dphi_total, 0.0))

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

# Potential
ax = axes[0]
ax.plot(r, phi_bulge, "--", color="steelblue", lw=1,
        label=f"bulge  ($M={M_bulge}$, $b={b_bulge}$)")
ax.plot(r, phi_halo,  "--", color="tomato",    lw=1,
        label=f"halo   ($M={M_halo}$, $b={b_halo}$)")
ax.plot(r, phi_total,       color="k",         lw=2,  label="total")
ax.set_xlabel(r"$r$")
ax.set_ylabel(r"$\Phi(r)$")
ax.set_title("Composite potential")
ax.legend(fontsize=8)

# Rotation-curve decomposition
ax = axes[1]
ax.plot(r, vc_bulge, "--", color="steelblue", lw=1, label="bulge")
ax.plot(r, vc_halo,  "--", color="tomato",    lw=1, label="halo")
ax.plot(r, vc_total,       color="k",         lw=2,  label="total")
ax.set_xlabel(r"$r$")
ax.set_ylabel(r"$v_c(r)$")
ax.set_title("Rotation-curve decomposition")
ax.legend(fontsize=8)

fig.tight_layout()
plt.show()

# Sanity check: superposition holds for the force
assert np.allclose(dphi_total, dphi_bulge + dphi_halo)
print("Superposition of forces verified ✓")


**Exercise 4.1** — Increase `M_halo` to 3.0 (keep `M_bulge = 0.2`).
How does the rotation curve change, particularly at large $r$?

**Exercise 4.2** — Add a third component: a disk-like Plummer with `b_disk = 0.05`,
`M_disk = 0.15`.  What does it contribute to $v_c$ at small radii?

**Exercise 4.3** — Compute $\rho_\text{total}(r) = \rho_\text{bulge}(r) + \rho_\text{halo}(r)$
and verify Poisson's equation with $\Phi_\text{total}$ using `numpy.gradient`.
